# PRM Demo: Build and Query a Roadmap

This notebook is designed as a self-learning walkthrough for PRM (Probabilistic Roadmap).

What you will learn:
- why PRM separates a "build roadmap" phase and a "query" phase,
- how sampling density and graph connectivity affect success,
- how shortest-path search is run on the sampled roadmap.

How to use this notebook:
1. Run each cell in order.
2. After each stage, inspect outputs before moving on.
3. At the end, change `num_samples` and `k_neighbors` and observe behavior changes.

In [ ]:
%matplotlib inline\n
\n
import math\n
import random\n
\n
import matplotlib.pyplot as plt\n
import networkx as nx\n
import numpy as np

## 1. Create an environment and define the query

In this section we create a 2D occupancy map with obstacles, then define `start` and `goal`.

Focus question:
- Is there at least one clear corridor from start to goal?

In [ ]:
H, W = 60, 60\n
grid = np.zeros((H, W), dtype=np.uint8)\n
\n
# Obstacles\n
grid[10:50, 20] = 1\n
grid[10:50, 40] = 1\n
grid[30, 20:41] = 1\n
grid[14, 20] = 0\n
grid[46, 40] = 0\n
grid[30, 30] = 0\n
\n
start = (4, 4)\n
goal = (55, 55)

## 2. Geometry and collision-check helpers

PRM needs local checks to connect nodes safely.

Key idea:
- `line_free(a, b)` acts as a local planner that approves or rejects an edge.

In [ ]:
def in_bounds(r, c):\n
    return 0 <= r < H and 0 <= c < W\n
\n
def free_cell(p):\n
    r, c = p\n
    return in_bounds(r, c) and grid[r, c] == 0\n
\n
def line_free(a, b):\n
    r0, c0 = a\n
    r1, c1 = b\n
    n = int(max(abs(r1 - r0), abs(c1 - c0))) + 1\n
    for i in range(n + 1):\n
        t = i / max(1, n)\n
        rr = int(round(r0 + t * (r1 - r0)))\n
        cc = int(round(c0 + t * (c1 - c0)))\n
        if not free_cell((rr, cc)):\n
            return False\n
    return True\n
\n
def dist(a, b):\n
    return math.hypot(a[0] - b[0], a[1] - b[1])

## 3. Sample free nodes and build the roadmap graph

This is the PRM "offline" stage.
- Sample valid points in free space.
- Connect each sample to nearby neighbors if the local edge is collision-free.

Watch:
- Higher `num_samples` improves connectivity but increases compute.
- Higher `k_neighbors` can reduce disconnected components.

In [ ]:
rng = random.Random(7)\n
num_samples = 320\n
k_neighbors = 10\n
\n
samples = [start, goal]\n
while len(samples) < num_samples + 2:\n
    p = (rng.randrange(H), rng.randrange(W))\n
    if free_cell(p):\n
        samples.append(p)\n
\n
G = nx.Graph()\n
for i, p in enumerate(samples):\n
    G.add_node(i, pos=p)\n
\n
for i, p in enumerate(samples):\n
    nbrs = sorted(((dist(p, q), j) for j, q in enumerate(samples) if j != i), key=lambda x: x[0])[:k_neighbors]\n
    for d, j in nbrs:\n
        q = samples[j]\n
        if line_free(p, q):\n
            G.add_edge(i, j, weight=d)\n
\n
print('Nodes:', G.number_of_nodes(), 'Edges:', G.number_of_edges())

## 4. Solve a shortest-path query on the roadmap

Now we switch to the "online query" stage: connect start and goal (already included as nodes) and run shortest-path search on the graph.

Interpretation tip:
- If this step fails, roadmap connectivity is usually the issue, not path-search logic.

In [ ]:
start_id, goal_id = 0, 1\n
path_ids = nx.shortest_path(G, source=start_id, target=goal_id, weight='weight')\n
path = [samples[i] for i in path_ids]\n
path_len = sum(dist(path[i], path[i + 1]) for i in range(len(path) - 1))\n
print('Path nodes:', len(path), 'Path length:', round(path_len, 2))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))\n
ax.imshow(grid, cmap='Greys', origin='upper')\n
\n
for u, v in G.edges():\n
    (r1, c1), (r2, c2) = samples[u], samples[v]\n
    ax.plot([c1, c2], [r1, r2], color='lightsteelblue', linewidth=0.5, alpha=0.6)\n
\n
px = [p[1] for p in path]\n
py = [p[0] for p in path]\n
ax.plot(px, py, color='deepskyblue', linewidth=2.5, label='PRM path')\n
ax.scatter(start[1], start[0], c='limegreen', s=90, marker='o', label='start')\n
ax.scatter(goal[1], goal[0], c='crimson', s=90, marker='*', label='goal')\n
ax.set_title('PRM roadmap and shortest path')\n
ax.set_xticks([])\n
ax.set_yticks([])\n
ax.legend(loc='upper right')\n
plt.tight_layout()\n
plt.show()